## Simple program to demonstrate the neutral behaviour of a time-delay system

refer MSSP 2025

Second-order form:

    mp xp''(t)  + cp (xp'(t) - xa'(t)) + kp (xp(t) - xa(t)) = ca(xa'(t)-xp'(t)) + ka(xa(t) - xp(t)) - u(t)
    ma xa''(t) + ca (xa'(t) - xp'(t)) + ka (xa(t) - xp(t)) = u(t)

    M = diag(mp,ma), 
    C = [   cp          -(cp+ca)    ]
        [   -(cp+ca)    ca          ]
    K = [   kp          -(kp+ka)    ]
        [   -(kp+ka)    ka          ]

Original system:

    x'(t)   = A0 x(t) + B1 u(t) + B2 fd(t)
    z(t)    = C2 x(t)

    A   =   [   0   I       ]
            [ -M\K  -M\D    ]
    B1  =   [   0   ]       C1 = [  I   ]
            [   Ba  ]       
    B2  =   [   0   ]       C2 = [  I   ]
            [   Bd  ]            
    D   =       0


In [1]:
# System parameters

# Masses
m0, m1, m2, ma = 1.1750, 0.5050, 0.7290, 0.5200

# Stiffness
k0, k1, k2, k3, ka, k4 = 1001, 749, 711, 950, 407, 377

# Damping
ca, c0, c1, c2, c3, c4 = 1.8000, 4.3500, 0.8500, 1.8500, 4.9500, 0

In [ ]:
import numpy as np
import tdspy as tds
from scipy.linalg import inv

""" generates rdde for the system described in the article
x'(t) = A x(t) + B2 f(t) + B1 u(t-tau)
y(t) = C2 x(t)
z(t) = C1 x(t)

with 
A = [   0   1   0   0   0   0   0   0;
        -2156.6 -6  637.4   0.7 320.9   0   346.4;  
        0   0   0   1   0   0   0   0;
        1483.2  1.7 -2891.1 -5.3    1407.9;
        0   0   0   0   0   1   0   0;
        517.1   0   975.3   2.5 -2795.6 -9.3    0;  
        0   0   0   0   0   0   0   1;
        782.7   3.5 0   0   0   0   0   -782.7      ]

B = [   0   0   0   0   0   1.3717  0   0   ]'

C = [   0   0   1   0   0   0   0   0   ]

D = 0

B1  = [  0   -0.8511 0   0   0   0   0   1.9231  ]'

C1  = [ 1   0   0   0   0   0   0   0   
        0   1   0   0   0   0   0   0
        0   0   0   0   1   0   0   0
        0   0   0   0   0   1   0   0
        0   0   0   0   0   0   1   0
        0   0   0   0   0   0   0   1   ]

"""
    
# Entering matrix entries

a21, a22, a23, a24 = -(k0+k1+ka+k4)/m0, -(c0+c1+ca+c4)/m0, k1/m0, c1/m0
a25, a26, a27, a28 = k4/m0, c4/m0, ka/m0, ca/m0

a41, a42, a43, a44 = k1/m1, c1/m1, -(k1+k2)/m1, -(c1+c2)/m1
a45, a46, a47, a48 = k2/m1, c2/m1, 0, 0

a61, a62, a63, a64 = k4/m2, c4/m2, k2/m2, c2/m2
a65, a66, a67, a68 = -(k2+k3+k4)/m2, -(c2+c3+c4)/m2, 0, 0

a81, a82, a83, a84 = ka/ma, ca/ma, 0, 0
a85, a86, a87, a88 = 0, 0, -ka/ma, -ca/ma

A0 = np.array([[    0,      1,      0,      0,      0,      0,      0,     0           ],
                [   a21,    a22,    a23,    a24,    a25,    a26,    a27,   a28         ],
                [   0,      0,      0,      1,      0,      0,      0,     0           ],
                [   a41,    a42,    a43,    a44,    a45,    a46,    a47,   a48         ],
                [   0,      0,      0,      0,      0,      1,      0,      0          ],
                [   a61,    a62,    a63,    a64,    a65,    a66,    a67,    a68        ],
                [   0,      0,      0,      0,      0,      0,      0,      1          ],
                [   a81,   a82,     a83,    a84,    a85,    a86,   a87,     a88        ]])
A = np.stack([A0], axis=2)

hA = np.array([0.])

B1 =  np.array([[   0,   0,   0,   0,   0,   0,     0,   0   ],
                [   0,   0,   0,   0,   0,   0,     0,   0   ],
                [   0,   0,   0,   0,   0,   1/m2,  0,   0   ]]).T    
B2 =  np.array([[   0,   -1/m0,   0,   0,   0,   0,     0,   1/ma], # u1 column
                [   0,   1/m0,   0,   0,   0,   0,     0,   0  ], # u2 column
                [   0,   0,   0,   0,   0,   0,     0,   0   ]]).T # d column
                
B = np.stack([B1, B2], axis=2)
hB = np.array([0.0, 0.0019])
C1 = np.array( [[ 1,   0,   0,   0,   0,   0,   0,   0   ],
                [ 0,   1,   0,   0,   0,   0,   0,   0   ],
                [ 0,   0,   0,   0,   1,   0,   0,   0   ],
                [ 0,   0,   0,   0,   0,   1,   0,   0   ],
                [ 0,   0,   0,   0,   0,   0,   1,   0   ],
                [ 0,   0,   0,   0,   0,   0,   0,   1   ],
                [ 0,   0,   1,   0,   0,   0,   0,   0   ]]) # the last row is z
C = np.stack([C1], axis=2)
hC = np.array([0])
D = np.zeros(shape=(7,3,1), dtype=float)
hD = np.array([0.])

rdde = tds.DDAE(A=A, hA=hA, B=B, hB=hB, C=C, hC=hC, D=D, hD=hD)

z,_ = tds.roots(rdde,r=-10)
print("z=",z)

z= [-4.99804115+63.98769822j -4.99804115-63.98769822j
 -3.64979113+53.69972212j -3.64979113-53.69972212j
 -1.0193508 +21.61010542j -1.0193508 -21.61010542j
 -2.37950007+33.59400171j -2.37950007-33.59400171j]
